# DFK Model → GGUF Converter (Fixed)
**PENTING:** Jalankan cell satu per satu. Jangan skip.
Cell 5 perlu waktu ~10-15 menit. File f16 harusnya ~17GB, Q4_K_M harusnya ~4.7GB.

In [ ]:
# Cell 1: Install dependency
import subprocess, sys, os
subprocess.run([sys.executable,'-m','pip','install','-q',
    'huggingface_hub','transformers','sentencepiece','gguf','numpy'], check=True)
print('✓ Done')

In [ ]:
# Cell 2: Clone llama.cpp
LLAMA_DIR  = '/content/llama.cpp'
MODEL_DIR  = '/content/dfk_model'
OUTPUT_F16 = '/content/model-f16.gguf'
OUTPUT_Q4  = '/content/model-q4_k_m.gguf'

if not os.path.exists(LLAMA_DIR):
    subprocess.run(['git','clone','--depth=1',
        'https://github.com/ggerganov/llama.cpp', LLAMA_DIR], check=True)
    subprocess.run([sys.executable,'-m','pip','install','-q',
        '-r', f'{LLAMA_DIR}/requirements.txt'], check=True)
print('✓ llama.cpp ready')

In [ ]:
# Cell 3: Download model (~35 GB)
from huggingface_hub import snapshot_download
HF_TOKEN = ''  # isi jika perlu
MODEL_ID = 'aitf-komdigi/KomdigiITS-8B-DFK-TextClassification'

if not os.path.exists(os.path.join(MODEL_DIR, 'config.json')):
    print(f'Downloading {MODEL_ID} (~35 GB)...')
    snapshot_download(repo_id=MODEL_ID, local_dir=MODEL_DIR,
        local_dir_use_symlinks=False, token=HF_TOKEN or None)
    print('✓ Download selesai')
else:
    print('✓ Model sudah ada')
print('Files:', sorted(os.listdir(MODEL_DIR)))

In [ ]:
# Cell 4: Patch config.json dan tokenizer_config.json
import json
config_path = os.path.join(MODEL_DIR, 'config.json')
with open(config_path) as f: cfg = json.load(f)
if cfg.get('model_type') in ('mistral3','ministral3'):
    cfg['model_type'] = 'mistral'
    cfg['architectures'] = ['MistralForCausalLM']
    print('Patched model_type')
if 'generation_config' in cfg:
    del cfg['generation_config']
    print('Removed generation_config')
with open(config_path,'w') as f: json.dump(cfg, f, indent=2)

tok_path = os.path.join(MODEL_DIR, 'tokenizer_config.json')
with open(tok_path) as f: tok = json.load(f)
if tok.get('tokenizer_class') == 'TokenizersBackend':
    tok['tokenizer_class'] = 'PreTrainedTokenizerFast'
    with open(tok_path,'w') as f: json.dump(tok, f, indent=2)
    print('Patched tokenizer_class')
print('✓ Patches done')

In [ ]:
# Cell 5: Convert ke GGUF f16
# Hapus file lama jika ada
if os.path.exists(OUTPUT_F16):
    os.remove(OUTPUT_F16)
    print('Removed old f16 file')

convert_script = os.path.join(LLAMA_DIR, 'convert_hf_to_gguf.py')
print(f'Converting {MODEL_DIR} → {OUTPUT_F16} ...')
print('(sekitar 10-15 menit)\n')

result = subprocess.run(
    [sys.executable, convert_script,
     MODEL_DIR, '--outfile', OUTPUT_F16, '--outtype', 'f16'],
    text=True, capture_output=True
)
if result.stdout: print('STDOUT:', result.stdout[-3000:])
if result.stderr: print('STDERR:', result.stderr[-3000:])

if result.returncode != 0:
    raise RuntimeError(f'Conversion GAGAL (exit {result.returncode}). Lihat STDERR di atas.')

size_gb = os.path.getsize(OUTPUT_F16) / 1e9
print(f'\n✓ f16 GGUF: {size_gb:.2f} GB')

# VALIDASI: f16 harus ≥ 15 GB
assert size_gb >= 15, f'ERROR: File terlalu kecil ({size_gb:.2f} GB)! Konversi tidak berhasil.'
print('✓ Ukuran valid (≥ 15 GB)')

In [ ]:
# Cell 6: Build llama-quantize dan quantize ke Q4_K_M
# Hapus file lama jika ada
if os.path.exists(OUTPUT_Q4):
    os.remove(OUTPUT_Q4)
    print('Removed old Q4 file')

quantize_bin = os.path.join(LLAMA_DIR, 'build', 'bin', 'llama-quantize')
if not os.path.exists(quantize_bin):
    print('Building llama-quantize...')
    build_dir = os.path.join(LLAMA_DIR, 'build')
    os.makedirs(build_dir, exist_ok=True)
    subprocess.run(['cmake','-B',build_dir,LLAMA_DIR,
        '-DLLAMA_NATIVE=OFF','-DCMAKE_BUILD_TYPE=Release'], check=True)
    subprocess.run(['cmake','--build',build_dir,
        '--target','llama-quantize','-j4'], check=True)
    print('✓ Built')

print('Quantizing f16 → Q4_K_M...')
subprocess.run([quantize_bin, OUTPUT_F16, OUTPUT_Q4, 'Q4_K_M'], check=True)

size_gb = os.path.getsize(OUTPUT_Q4) / 1e9
print(f'\n✓ Q4_K_M GGUF: {size_gb:.2f} GB')

# VALIDASI: Q4_K_M harus ≥ 4 GB
assert size_gb >= 4, f'ERROR: File terlalu kecil ({size_gb:.2f} GB)! Quantization tidak berhasil.'
print('✓ Ukuran valid (≥ 4 GB)')

os.remove(OUTPUT_F16)
print('✓ f16 dihapus')

In [ ]:
# Cell 7: Upload ke HuggingFace Hub (REPLACE file lama yang corrupt)
from huggingface_hub import HfApi

HF_TOKEN      = ''  # WAJIB isi token HF kamu
GGUF_REPO     = 'ggapar/KomdigiITS-8B-DFK-GGUF'
GGUF_FILENAME = 'model-q4_k_m.gguf'

assert HF_TOKEN, 'Isi HF_TOKEN dulu!'
assert os.path.exists(OUTPUT_Q4), 'Jalankan cell 5-6 dulu'

size_gb = os.path.getsize(OUTPUT_Q4) / 1e9
print(f'File: {OUTPUT_Q4} ({size_gb:.2f} GB)')
assert size_gb >= 4, 'File terlalu kecil, jangan upload!'

api = HfApi(token=HF_TOKEN)
api.create_repo(GGUF_REPO, repo_type='model', exist_ok=True)

print(f'Uploading {size_gb:.2f} GB ke {GGUF_REPO}...')
api.upload_file(
    path_or_fileobj=OUTPUT_Q4,
    path_in_repo=GGUF_FILENAME,
    repo_id=GGUF_REPO,
    repo_type='model',
    commit_message=f'Upload DFK GGUF Q4_K_M ({size_gb:.1f} GB)',
)
print(f'\n✓ Upload selesai!')
print(f'URL: https://huggingface.co/{GGUF_REPO}')
print(f'Verifikasi: file harus ~{size_gb:.1f} GB di repo')